### ***Installing the required dependencies***

In [1]:
!pip install transformers datasets peft accelerate evaluate sentencepiece sacrebleu nltk bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.1 MB/s eta 0:00:00


In [2]:
!pip install -q -U transformers datasets peft accelerate evaluate sentencepiece sacrebleu nltk bert_score
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 55.1 MB/s eta 0:00:00


### ***Loading the dataset for English to Spanish translation task***

In [1]:
from datasets import load_dataset

# Newer `datasets` versions require the full "namespace/name" id.
# The bare "opus_books" no longer works -> use "Helsinki-NLP/opus_books".
dataset = load_dataset("Helsinki-NLP/opus_books", "en-es")
dataset = dataset["train"].train_test_split(test_size=0.2)

print(dataset)

README.md:   0%|          | 0.00/28.1k [00:00<?, ?B/s]

en-es/train-00000-of-00001.parquet:   0%|          | 0.00/16.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/93470 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 74776
    })
    test: Dataset({
        features: ['id', 'translation'],
        num_rows: 18694
    })
})


### ***Loading pre-trained model and tokenizer from Hugging Face Model Hub***

In [2]:
from transformers import AutoTokenizer

In [3]:
# Switched to a WEAK base model (t5-small) so fine-tuning shows a clear improvement.
# (opus-mt is already an expert translator, so fine-tuning could only hurt it.)
# t5-small was NOT pre-trained on English->Spanish, so "before" will be poor
# and "after" should be noticeably better.
model_checkpoint = "t5-small"           # alternative: "google/mt5-small" (better Spanish, but slower)
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

### ***Preprocessing function to prepare the dataset for the model***

In [4]:
# T5 is a multi-task model: it needs a task PREFIX telling it what to do.
prefix = "translate English to Spanish: "

def preprocess_function(examples):
    inputs  = [prefix + ex["en"] for ex in examples["translation"]]
    targets = [ex["es"] for ex in examples["translation"]]

    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")
    # text_target is the correct, modern way to tokenize the labels for seq2seq
    labels = tokenizer(text_target=targets, max_length=128, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/74776 [00:00<?, ? examples/s]

Map:   0%|          | 0/18694 [00:00<?, ? examples/s]

### ***Load Pretrained Model***

In [6]:
from transformers import AutoModelForSeq2SeqLM

In [7]:
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

### ***Apply PEFT + LoRA Configuration***

In [8]:
from peft import LoraConfig, TaskType

In [9]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q", "v"],   # T5 attention layers are named q, k, v, o (NOT q_proj/v_proj)
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

In [10]:
from peft import get_peft_model

In [11]:
peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()

trainable params: 294,912 || all params: 60,801,536 || trainable%: 0.4850


### ***Training Arguments***

In [12]:
from transformers import Seq2SeqTrainingArguments
import torch

In [13]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=1e-3,                 # higher LR: the model must LEARN a new task here
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    save_strategy="epoch",
    predict_with_generate=True,
    generation_max_length=128,
    fp16=False,                         # IMPORTANT: T5 + fp16 often gives NaN loss; keep it off
    load_best_model_at_end=True,        # keep the best checkpoint...
    metric_for_best_model="bleu",       # ...judged by BLEU, not loss
    greater_is_better=True,
)

### ***Data Collator***

In [14]:
from transformers import DataCollatorForSeq2Seq

In [15]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=peft_model)

### ***Evaluation Metric: BLEU***

In [16]:
import evaluate
import numpy as np
import nltk

# METEOR needs these NLTK resources
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("punkt")

# Load the metrics once (more efficient than reloading every eval step)
bleu_metric   = evaluate.load("bleu")     # word n-gram overlap
chrf_metric   = evaluate.load("chrf")     # character n-gram (good for Spanish morphology)
meteor_metric = evaluate.load("meteor")   # synonyms + word order

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [17]:
def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # predict_with_generate can return a tuple; take the first element
    if isinstance(preds, tuple):
        preds = preds[0]

    preds  = np.array(preds)
    labels = np.array(labels)

    vocab_size = len(tokenizer)
    # Replace ANY invalid token id (-100 or out-of-vocab) with pad token id
    # -> this is what fixes the OverflowError during decoding
    preds  = np.where((preds >= 0) & (preds < vocab_size), preds, tokenizer.pad_token_id)
    labels = np.where((labels >= 0) & (labels < vocab_size), labels, tokenizer.pad_token_id)

    decoded_preds  = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    refs_nested = [[ref] for ref in decoded_labels]

    bleu_result   = bleu_metric.compute(predictions=decoded_preds, references=refs_nested)
    chrf_result   = chrf_metric.compute(predictions=decoded_preds, references=refs_nested)
    meteor_result = meteor_metric.compute(predictions=decoded_preds, references=decoded_labels)

    return {
        "bleu":   round(bleu_result["bleu"], 4),
        "chrf":   round(chrf_result["score"], 4),
        "meteor": round(meteor_result["meteor"], 4),
    }

### ***Trainer Setup***

In [18]:
from transformers import Seq2SeqTrainer

In [19]:
trainer = Seq2SeqTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

### ***Start Training***

In [20]:
trainer.train()

Epoch,Training Loss,Validation Loss,Bleu,Chrf,Meteor
1,1.237737,1.078662,0.040200,23.903000,0.215200
2,1.185290,1.034425,0.051600,25.874100,0.238200
3,1.175457,1.018656,0.055200,26.534600,0.241500


TrainOutput(global_step=14022, training_loss=1.2309560208530776, metrics={'train_runtime': 10000.3692, 'train_samples_per_second': 22.432, 'train_steps_per_second': 1.402, 'total_flos': 7641047500849152.0, 'train_loss': 1.2309560208530776, 'epoch': 3.0})

### ***Saving the fine-tuned model and tokenizer***

In [21]:
# Colab's working directory is /content
peft_model.save_pretrained('/content/fine-tuned-model')
tokenizer.save_pretrained('/content/fine-tuned-tokenizer')

('/content/fine-tuned-tokenizer/tokenizer_config.json',
 '/content/fine-tuned-tokenizer/tokenizer.json')

### ***Loading the fine-tuned model and tokenizer for inference***

In [22]:
from peft import PeftModel

# NOTE: peft_model.save_pretrained() saves ONLY the LoRA adapter,
# so we must reload the original base model first, then attach the adapter.
tokenizer = AutoTokenizer.from_pretrained('./fine-tuned-tokenizer')

base_for_inference = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
model = PeftModel.from_pretrained(base_for_inference, './fine-tuned-model')

# (Optional) merge the LoRA weights into the base model for faster inference
model = model.merge_and_unload()
model.eval()

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

### ***Example input text in English***

In [23]:
input_text = "Hello, how are you today?"

In [24]:
# Tokenize the input text (T5 needs the same translation prefix used in training)
prefix = "translate English to Spanish: "
inputs = tokenizer(prefix + input_text, return_tensors="pt", padding=True, truncation=True, max_length=128)

In [25]:
# Generate the translation (English to Spanish)
translated_tokens = model.generate(inputs["input_ids"], max_length=40, num_beams=4, early_stopping=True)

In [26]:
# Decode the translated tokens back to text (Spanish)
translated_text = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)

In [27]:
translated_text

'Por qué tiene ahora?'

### ***Zipping the model & tokenizer and downloading them in the browser (Google Colab)***

In [28]:
# Colab: these let us zip the folders and trigger a browser download
from google.colab import files
import shutil

In [29]:
# Zip the model folder, then download it straight to your computer
shutil.make_archive('/content/fine-tuned-model', 'zip', '/content', 'fine-tuned-model')
files.download('/content/fine-tuned-model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [30]:
# Zip the tokenizer folder, then download it straight to your computer
shutil.make_archive('/content/fine-tuned-tokenizer', 'zip', '/content', 'fine-tuned-tokenizer')
files.download('/content/fine-tuned-tokenizer.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
import torch
torch.cuda.empty_cache()  # Clears unused GPU memory

### ***Concept: Comparing translation quality BEFORE vs AFTER fine-tuning***

To actually *see* what the LoRA fine-tuning achieved, we translate the same set of
English sentences with two models:

1. **Before** — the original pre-trained `t5-small` (which was never trained on English→Spanish, so it translates poorly).
2. **After**  — our LoRA fine-tuned model.

Because the base model is weak at this task, fine-tuning should produce a **clear, positive** improvement here.
We also compute a **BERTScore** (a semantic / meaning-based metric) to judge quality beyond simple word overlap.

In [32]:
import torch

# A few sample English sentences to translate
sample_sentences = [
    "Hello, how are you today?",
    "The book is on the table.",
    "She walked slowly through the quiet garden.",
    "We are learning how to fine-tune a translation model.",
    "Knowledge is power, but wisdom is knowing how to use it.",
]

prefix = "translate English to Spanish: "

def translate(model, tokenizer, sentences, max_length=40, num_beams=4):
    """Translate English sentences to Spanish with the given model (adds the T5 prefix)."""
    model.eval()
    outputs = []
    for text in sentences:
        inputs = tokenizer(prefix + text, return_tensors="pt", padding=True,
                           truncation=True, max_length=128)
        with torch.no_grad():
            generated = model.generate(
                inputs["input_ids"],
                max_length=max_length,
                num_beams=num_beams,
                early_stopping=True,
            )
        outputs.append(tokenizer.decode(generated[0], skip_special_tokens=True))
    return outputs

# ---- BEFORE: original t5-small (no fine-tuning) ----
before_tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
before_model     = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
before_outputs   = translate(before_model, before_tokenizer, sample_sentences)

# ---- AFTER: our LoRA fine-tuned model (loaded above as `model`) ----
after_outputs = translate(model, tokenizer, sample_sentences)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [33]:
# Print a side-by-side comparison
for i, eng in enumerate(sample_sentences):
    print(f"English      : {eng}")
    print(f"Before (base): {before_outputs[i]}")
    print(f"After  (LoRA): {after_outputs[i]}")
    print("-" * 70)

English      : Hello, how are you today?
Before (base): Hallo, wie sind Sie heute?
After  (LoRA): Por qué tiene ahora?
----------------------------------------------------------------------
English      : The book is on the table.
Before (base): Das Buch ist auf dem Tisch.
After  (LoRA): El bucho está en la mesa.
----------------------------------------------------------------------
English      : She walked slowly through the quiet garden.
Before (base): Sie ging langsam durch den ruhigen Garten.
After  (LoRA): Y llevó lentamente en el jardin tranquilo.
----------------------------------------------------------------------
English      : We are learning how to fine-tune a translation model.
Before (base): Spanisch lernen wir, wie man ein Übersetzungsmodell anpasst.
After  (LoRA): En eso, el agua como reglar un modelo de traducción.
----------------------------------------------------------------------
English      : Knowledge is power, but wisdom is knowing how to use it.
Before (base

### ***Semantic check with BERTScore (before vs after)***

Note: a true BLEU/chrF comparison needs reference (gold) translations. Here we use
**BERTScore** to measure how semantically close each model's output is. If you have
gold Spanish references for these sentences, plug them into `references` below for a
fair before/after score.

In [34]:
# Example gold (reference) Spanish translations for the sample sentences.
# Replace these with verified human translations for a fair comparison.
references = [
    "Hola, ¿cómo estás hoy?",
    "El libro está sobre la mesa.",
    "Ella caminó lentamente por el jardín tranquilo.",
    "Estamos aprendiendo a afinar un modelo de traducción.",
    "El conocimiento es poder, pero la sabiduría es saber cómo usarlo.",
]

bertscore = evaluate.load("bertscore")

before_bs = bertscore.compute(predictions=before_outputs, references=references, lang="es")
after_bs  = bertscore.compute(predictions=after_outputs,  references=references, lang="es")

print("BERTScore F1 (higher = closer in meaning to reference)")
print(f"Before fine-tuning : {np.mean(before_bs['f1']):.4f}")
print(f"After  fine-tuning : {np.mean(after_bs['f1']):.4f}")

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERTScore F1 (higher = closer in meaning to reference)
Before fine-tuning : 0.8105
After  fine-tuning : 0.8249
